<h2>Joining Sequence/metadata Information with Interaction datset</h2>
<h6>We keep only interaction with avaiable sequences in both Interacting RNAs</h6>

In [ ]:
import pandas as pd

# load interactions
df_filt = pd.read_pickle("data/interactions_processed_noSeq.pkl")
rna_entities_ncRNA = pd.read_pickle("data/nc_rna_entities_processed.pkl")
rna_entities_mRNA = pd.read_pickle("data/m_rna_entities_processed.pkl") # ADD FILE-NAME

In [51]:
print(len(df_filt))
print(len(rna_entities_ncRNA))
print(len(rna_entities_mRNA))

9429404
40911
197490


In [ ]:
import pandas as pd

# ----------------------------
# Step 0: Ensure metadata columns exist
# ----------------------------
for col in ['Species1_TaxID', 'Species2_TaxID', 'Category1_std', 'Category2_std']:
    if col not in df_filt.columns:
        df_filt[col] = None
print("Step 0: Metadata columns ensured.")

# ----------------------------
# Step 1: Normalize Raw_ID lists to tuples
# ----------------------------
def normalize_raw_ids(raw_ids):
    if isinstance(raw_ids, list):
        return tuple(raw_ids)
    return raw_ids

df_filt['Raw_ID1_key'] = df_filt['Raw_ID1_list'].apply(normalize_raw_ids)
df_filt['Raw_ID2_key'] = df_filt['Raw_ID2_list'].apply(normalize_raw_ids)

print("Step 1: Sample normalized Raw_ID keys")
print(df_filt[['Raw_ID1_key', 'Raw_ID2_key']].head())

# ----------------------------
# Step 2: Extract unique RNA entities from interactions
# ----------------------------
def extract_entities(df, interactor_prefix):
    cols = [f"{interactor_prefix}.Symbol", f"Category{interactor_prefix[-1]}", f"Species{interactor_prefix[-1]}_TaxID", f"Raw_ID{interactor_prefix[-1]}_key"]
    rna_entities = df[cols].rename(columns={
        f"{interactor_prefix}.Symbol": "Symbol",
        f"Category{interactor_prefix[-1]}": "Category",
        f"Species{interactor_prefix[-1]}_TaxID": "Species",
        f"Raw_ID{interactor_prefix[-1]}_key": "Raw_ID_key"
    })
    rna_entities['RNA_type'] = rna_entities['Category']
    rna_entities['RNA_IDs'] = rna_entities['Raw_ID_key']
    return rna_entities.drop_duplicates()

rna_entities_1 = extract_entities(df_filt, 'Interactor1')
rna_entities_2 = extract_entities(df_filt, 'Interactor2')

# Combine and drop duplicates
rna_entities_all = pd.concat([rna_entities_1, rna_entities_2], ignore_index=True)

print(f"Step 2: Unique RNA entities extracted from interactions: {len(rna_entities_all)}")
print(rna_entities_all.head())

# ----------------------------
# Step 3: Add sequences for ncRNA and mRNA
# ----------------------------
# Ensure Raw_ID_key in mRNA entities is tuple
rna_entities_mRNA['Raw_ID_key'] = rna_entities_mRNA['RNA_IDs'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
# Merge ncRNA and mRNA
rna_entities_all = pd.concat([rna_entities_all, rna_entities_mRNA[['Symbol','Category','Species','Raw_ID_key','sequences']]], ignore_index=True)
print(f"Step 3: Total RNA entities with sequences: {rna_entities_all['sequences'].notna().sum()}")
print(rna_entities_all.head())

# ----------------------------
# Step 4: Create canonical entity_key
# ----------------------------
rna_entities_all['entity_key'] = rna_entities_all.apply(
    lambda row: (row['Symbol'], row['Category'], row['Species'], row['Raw_ID_key']),
    axis=1
)
print("Step 4: Sample entity_key")
print(rna_entities_all[['Symbol','entity_key']].head())

# ----------------------------
# Step 5: Create interaction keys
# ----------------------------
df_filt['key1'] = df_filt.apply(lambda row: (row['Interactor1.Symbol'], row['Category1_std'] or row['Category1'], row['Species1_TaxID'], row['Raw_ID1_key']), axis=1)
df_filt['key2'] = df_filt.apply(lambda row: (row['Interactor2.Symbol'], row['Category2_std'] or row['Category2'], row['Species2_TaxID'], row['Raw_ID2_key']), axis=1)

print("Step 5: Sample interaction keys")
print(df_filt[['key1','key2']].head())

# ----------------------------
# Step 6: Merge sequences onto interactions
# ----------------------------
df_merged = df_filt.merge(
    rna_entities_all[['entity_key','sequences']],
    left_on='key1', right_on='entity_key', how='left'
).rename(columns={'sequences':'seq1_list'}).drop(columns=['entity_key'])

df_merged = df_merged.merge(
    rna_entities_all[['entity_key','sequences']],
    left_on='key2', right_on='entity_key', how='left'
).rename(columns={'sequences':'seq2_list'}).drop(columns=['entity_key'])

print("Step 6: Sample merged dataframe with sequences")
print(df_merged[['key1','seq1_list','key2','seq2_list']].head())

# ----------------------------
# Step 7: Filter interactions with sequences for both partners
# ----------------------------
has_seq1 = df_merged['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_seq2 = df_merged['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
df_filtered = df_merged[has_seq1 & has_seq2]

print(f"Step 7: Total interactions before filtering: {len(df_merged)}")
print(f"Step 7: Total interactions after filtering (both partners with sequences): {len(df_filtered)}")

# ----------------------------
# Step 8: Count interactions involving at least one mRNA
# ----------------------------
def contains_mrna(key):
    cat = key[1]
    return cat.lower() in ['mrna','protein_coding']

df_filtered['interactor1_mRNA'] = df_filtered['key1'].apply(contains_mrna)
df_filtered['interactor2_mRNA'] = df_filtered['key2'].apply(contains_mrna)

interactions_with_mRNA = df_filtered[df_filtered['interactor1_mRNA'] | df_filtered['interactor2_mRNA']]
interactions_no_mRNA = df_filtered[~(df_filtered['interactor1_mRNA'] | df_filtered['interactor2_mRNA'])]

print(f"Interactions involving at least one mRNA: {len(interactions_with_mRNA)}")
print(f"Interactions with no mRNA: {len(interactions_no_mRNA)}")

Step 0: Metadata columns ensured.


In [53]:
print("Interactions matched to interactor1:", df_filt_copy['key1'].isin(rna_entities_final['entity_key']).sum())
print("Interactions matched to interactor2:", df_filt_copy['key2'].isin(rna_entities_final['entity_key']).sum())

Interactions matched to interactor1: 7374955
Interactions matched to interactor2: 233788


In [66]:
# Check for mRNA in interactor1 or interactor2
is_mrna1 = df_filt_final['Category1'].eq('mRNA')
is_mrna2 = df_filt_final['Category2'].eq('mRNA')

# Interactions involving at least one mRNA
interactions_with_mrna = df_filt_final[is_mrna1 | is_mrna2]
interactions_without_mrna = df_filt_final[~(is_mrna1 | is_mrna2)]

print(f"Total interactions: {len(df_filt_final)}")
print(f"Interactions involving at least one mRNA: {len(interactions_with_mrna)}")
print(f"Interactions with no mRNA: {len(interactions_without_mrna)}")

Total interactions: 119381
Interactions involving at least one mRNA: 0
Interactions with no mRNA: 119381


# todo: refactor

In [21]:
df_filt_final.to_pickle("interactions_with_sequences_RNA_RNA_13-03-2026.pkl")

In [100]:
import pandas as pd

# Collect all entity_keys present in the filtered interactions
keys_retained = pd.concat([
    df_filt_final['key1'],
    df_filt_final['key2']
]).drop_duplicates()

print(f"Unique RNA entities retained: {len(keys_retained)}")


Unique RNA entities retained: 11929


In [101]:
def count_interactions_under_length(df, max_len):

    kept_rows = []

    for _, row in df.iterrows():

        seq1 = row['seq1_list']
        seq2 = row['seq2_list']

        if not isinstance(seq1, list) or not isinstance(seq2, list):
            continue

        valid_1 = any(len(s) <= max_len for s in seq1)
        valid_2 = any(len(s) <= max_len for s in seq2)

        if valid_1 and valid_2:
            kept_rows.append(row)

    df_kept = pd.DataFrame(kept_rows)

    total_rows = len(df_kept)

    print(f"\nMax length threshold: {max_len} nt")
    print(f"Total interactions: {total_rows}")

    # ----------------------------
    # Row-level coverage
    # ----------------------------
    count_with_seq = (
        df_kept['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0) &
        df_kept['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
    ).sum()

    def has_valid_list(x):
        return isinstance(x, list) and any(pd.notna(v) and v != "" for v in x)
    
    # Masks per side
    go1 = df_kept['GO1'].apply(has_valid_list)
    go2 = df_kept['GO2'].apply(has_valid_list)
    
    rfam1 = df_kept['Rfam1'].apply(has_valid_list)
    rfam2 = df_kept['Rfam2'].apply(has_valid_list)
    
    # BOTH sides annotated
    count_go_both = (go1 & go2).sum()
    count_rfam_both = (rfam1 & rfam2).sum()
    
    # AT LEAST ONE side annotated
    count_with_go = (go1 | go2).sum()
    count_with_rfam = (rfam1 | rfam2).sum()

    print(f"Interactions with sequences (both sides): {count_with_seq} ({count_with_seq/total_rows:.2%})")
    print(f"Interactions with GO terms (both sides): {count_go_both} ({count_go_both/total_rows:.2%})")
    print(f"Interactions with GO terms (one side at least): {count_with_go} ({count_with_go/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (both sides): {count_rfam_both} ({count_rfam_both/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (one side at least): {count_with_rfam} ({count_with_rfam/total_rows:.2%})")

    # ----------------------------
    # Sequence-level statistics
    # ----------------------------

    total_sequences = 0
    all_sequences = set()

    for col in ['seq1_list', 'seq2_list']:
        for seq_list in df_kept[col]:
            if isinstance(seq_list, list):
                total_sequences += len(seq_list)
                all_sequences.update(seq_list)

    total_unique_sequences = len(all_sequences)

    print(f"Total sequences (with redundancy): {total_sequences}")
    print(f"Total unique sequences: {total_unique_sequences}")

    if total_sequences > 0:
        redundancy = 1 - total_unique_sequences / total_sequences
        print(f"Redundancy ratio: {redundancy:.2%}")
    else:
        print("Redundancy ratio: NA")

    keys_retained = pd.concat([
        df_kept['key1'],
        df_kept['key2']
    ]).drop_duplicates()
    
    print(f"Unique RNA entities retained: {len(keys_retained)}")

    return df_kept



In [102]:
df_1000000 = count_interactions_under_length(df_filt_final, 1000000)


Max length threshold: 1000000 nt
Total interactions: 82171
Interactions with sequences (both sides): 82171 (100.00%)
Interactions with GO terms (both sides): 6314 (7.68%)
Interactions with GO terms (one side at least): 18164 (22.11%)
Interactions with Rfam IDs (both sides): 6584 (8.01%)
Interactions with Rfam IDs (one side at least): 21230 (25.84%)
Total sequences (with redundancy): 439193
Total unique sequences: 26994
Redundancy ratio: 93.85%
Unique RNA entities retained: 11929


In [104]:
df_10000 = count_interactions_under_length(df_filt_final, 10000)
df_4096 = count_interactions_under_length(df_filt_final, 4096)
df_2048 = count_interactions_under_length(df_filt_final, 2048)
df_1024 = count_interactions_under_length(df_filt_final, 1024)


Max length threshold: 10000 nt
Total interactions: 81458
Interactions with sequences (both sides): 81458 (100.00%)
Interactions with GO terms (both sides): 6293 (7.73%)
Interactions with GO terms (one side at least): 17866 (21.93%)
Interactions with Rfam IDs (both sides): 6537 (8.02%)
Interactions with Rfam IDs (one side at least): 20671 (25.38%)
Total sequences (with redundancy): 436769
Total unique sequences: 26819
Redundancy ratio: 93.86%
Unique RNA entities retained: 11849

Max length threshold: 4096 nt
Total interactions: 75458
Interactions with sequences (both sides): 75458 (100.00%)
Interactions with GO terms (both sides): 6113 (8.10%)
Interactions with GO terms (one side at least): 15832 (20.98%)
Interactions with Rfam IDs (both sides): 6349 (8.41%)
Interactions with Rfam IDs (one side at least): 18417 (24.41%)
Total sequences (with redundancy): 409189
Total unique sequences: 25054
Redundancy ratio: 93.88%
Unique RNA entities retained: 10927

Max length threshold: 2048 nt
Tota